<a href="https://colab.research.google.com/github/chris-misa/webcam-yolo/blob/main/train.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Train yolo models to fine-tune on custom datasets.

First, mount google drive using "Files" tab on left.

Next, set the variables in the follow cell.

Finally, run the rest of the cells.

In [ ]:
# Path to zip file containing data to train on.
# Assumes the zip file was exported in YOLO format from label-studio.
# Hint: find the file in the "Files" tab, then right-click on it and select "Copy path"...
ZIP_DATA_PATH="/content/drive/MyDrive/YOLO/project-3-at-2026-01-08-05-33-e2277f80.zip"

# Class names of the training data.
CLASSES = ["brush"]

# Pre-trained model to load (or None to start a new model)
#IN_MODEL = None
IN_MODEL = "/content/drive/MyDrive/YOLO/brush-test1/weights/best.pt"

# Name to give the resulting fine-tuned model.
MODEL_NAME="brush-test1"

# Path to write output to (will append MODEL_NAME).
OUTPUT_PATH="/content/drive/MyDrive/YOLO/"

Install dependencies, unzip the data to local storage...

In [ ]:
!pip install ultralytics
!rm -rf /content/tmp
!mkdir -p /content/tmp
!unzip {ZIP_DATA_PATH} -d /content/tmp

Import the YOLO from ultralytics

In [ ]:
import torch
from ultralytics import YOLO

In [ ]:
torch.cuda.is_available()

Create the model either from a generic base version or the already-trained model pointed to by IN_MODEL.

Also, write out a config file for training.

In [ ]:
model = YOLO("yolo26n.pt" if IN_MODEL is None else IN_MODEL)
with open("conf.yaml", "w") as f:
  f.write(f"""
path: ./
train: tmp/images/
val: tmp/images/
test:

names:
""")
  for i, l in enumerate(CLASSES):
    f.write(f"  {i}: {l}\n")

  f.write("\n")

This next cell actually runs the training...

In [ ]:
res = model.train(
    data = "conf.yaml",
    imgsz = 640,
    epochs = 3,
    batch = 8,
    name = MODEL_NAME,
    exist_ok = True
)

Write the outputs from training back to the OUTPUT_PATH.

In [ ]:
!cp -r /content/runs/detect/{MODEL_NAME} /content/{MODEL_NAME}
!rm -rf {OUTPUT_PATH}/{MODEL_NAME}
!mv /content/{MODEL_NAME} {OUTPUT_PATH}/{MODEL_NAME}